# INF201 - Deliverable 4
## *Temperatures and Fractions*

**Group:** G6_2: Nikolai Friis, Tommy Rustad Apelseth, Aron Lerdahl

**GitHub repository:** https://github.com/tommyra06/INF201-Deliverables

---

### Contents
* Setup
* Task 1: *Temperatures*
* Task 2: *Fractions*
* Use of generative AI

---

## Setup

### Virtual environment

Create a new virtual environment (`.venv`). Install the packages for Deliverable 4, listed in `requirements.txt`. Use the command pallete.

### Python scripts
The classes are written in their own scripts so the tests can import them with `pytest`:

* `temperature_reader.py` - *the `Temperature` class (Task 1)*
* `test_temperature.py` - *pytest tests for the `Temperature` class (Task 1)*
* `fraction.py` - *the `Fraction` class (Task 2)*
* `test_fraction.py` - *pytest tests for the `Fraction` class (Task 2)*

### Imports
All imports used in the notebook are collected in the cell below.

In [ ]:
# Library imports that come with Python
# import ...  # TODO: What it is used for (Task X)

# Our own classes, from the scripts in this folder
# from temperature_reader import Temperature  # Task 1
# from fraction import Fraction  # Task 2

---

## Task 1: *Temperatures*

For Task 1, we have created the `Temperature` class that supports Celsius, Fahrenheit and Kelvin.

* `to_celsius`, `to_fahrenheit`, `to_kelvin` converts temperature to C, F or K
* `convert_to` activates the `to_XXXX` methods based of the target unit
* `__repr__`, `__str__` generates string representations of the class
* `__eq__`, `__ne__`, `__lt__`, `__le__`, `__gt__`, `__ge__`,  methods to compare with another object

In [ ]:
class Temperature:
    """Class that represents temperature with unit"""

    def __init__(self, value: float, unit: str):
        """
                Initialise class
        
                Args:
                    value (float): The temperature value
                    unit (str): The temperature unit
                """
        unit = unit.upper()
        absolue_zero = {
            'K': 0, 
            'C': -273.15, 
            'F': -459.67
            }
        
        if unit not in absolue_zero or value < self.absolue_zero[unit] - 1e-9:
            raise ValueError('Invalid unit')
        #Checks if given unit is valid and if given value is above absolute zero
        
        self._value = value
        self._unit = unit

    def to_celsius(self) -> float:
        """Converts temperature to celsius"""
        if self._unit == 'F':
            return((self._value - 32) * (5 / 9))
        elif self._unit == 'K':
            return(self._value - 273.15)
        else:
            return(self._value)

    def to_fahrenheit(self) -> float:
        """Converts temperature to fahrenheit"""
        if self._unit == 'C':
            return((self._value * (9 / 5)) + 32)
        elif self._unit == 'K':
            return((self._value - 273.15) * (9 / 5) + 32)
        else:
            return(self._value)

    def to_kelvin(self) -> float:
        """Converts temperature to kelvin"""
        if self._unit == 'C':
            return(self._value + 273.15)
        elif self._unit == 'F':
            return((self._value - 32) * (5 / 9) + 273.15)
        else:
            return(self._value)

    def convert_to(self, target_unit: str) -> object:
        """Creates new object with converted temperature to given unit"""
        converters = {
            'K': self.to_kelvin, 
            'F': self.to_fahrenheit, 
            'C': self.to_celsius
            }
        target_unit = target_unit.upper()

        if target_unit not in converters:
            raise ValueError('Invalid unit')
        
        return Temperature(converters[target_unit](), target_unit)

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Temperature):
            return NotImplemented
        return abs(self.to_kelvin() - other.to_kelvin()) <= 1e-9

    def __ne__(self, other: object) -> bool:
        if not isinstance(other, Temperature):
            return NotImplemented
        return not self.__eq__(other)

    def __lt__(self, other: object) -> bool:
        if not isinstance(other, Temperature):
            return NotImplemented
        return self.to_kelvin() < other.to_kelvin()

    def __le__(self, other: object) -> bool:
        if not isinstance(other, Temperature):
            return NotImplemented
        return self.to_kelvin() <= other.to_kelvin()

    def __gt__(self, other: object) -> bool:
        if not isinstance(other, Temperature):
            return NotImplemented
        return other < self

    def __ge__(self, other: object) -> bool:
        if not isinstance(other, Temperature):
            return NotImplemented
        return other <= self
    
    def __repr__(self) -> str:
        return(f'Temperature({self._value}, {self._unit})')

    def __str__(self) -> str:
        degree_sign = '\N{DEGREE SIGN}'
        return(f'{self._value:.2f} {degree_sign + self._unit}')

    @property
    def value(self) -> float:
        return self._value

    @property
    def unit(self) -> str:
        return self._unit



        

---

### Testing with pytest

Tests for a lot of things
1. If class was initialized correctly
2. If attributes are read_only
3. If unit and temperature valid are correct
4. Different conversions
5. Roundtrip of all conversions
6. All comparisons
7. if str and repr methods are correctly formatted

In [ ]:
!pytest -v test_temperature.py

def test_initialization():
    temp = Temperature(25, 'C')
    assert temp.value == 25
    assert temp.unit == 'C'

def test_read_only_properties():
    temp = Temperature(25, 'C')

    with pytest.raises(AttributeError):
        temp.value = 30

    with pytest.raises(AttributeError):
        temp.unit = 'F'

def test_validation():
    with pytest.raises(ValueError):
        Temperature(10, 'W')

    with pytest.raises(ValueError):
        Temperature(-1, 'K')

    with pytest.raises(ValueError):
        Temperature(-274, 'C')

    with pytest.raises(ValueError):
        Temperature(-460, 'F')

def test_conversions():
    assert Temperature(-273.15, 'C').to_fahrenheit() == pytest.approx(-459.67)
    assert Temperature(-459.67, 'F').to_kelvin() == pytest.approx(0)

    assert Temperature(-40, 'F').to_kelvin() == pytest.approx(233.15)
    assert Temperature(233.15, 'K').to_celsius() == pytest.approx(-40)

    assert Temperature(273.15, 'K').to_fahrenheit() == pytest.approx(32)
    assert Temperature(32, 'F').to_celsius() == pytest.approx(0)

    assert Temperature(37, 'C').to_kelvin() == pytest.approx(310.15)
    assert Temperature(310.15, 'K').to_fahrenheit() == pytest.approx(98.6)

    assert Temperature(100, 'C').to_fahrenheit() == pytest.approx(212)
    assert Temperature(212, 'F').to_kelvin() == pytest.approx(373.15)

def test_roundtrip():
    temp = Temperature(10, 'C')
    converted = temp.convert_to('F')
    converted = temp.convert_to('K')
    converted = temp.convert_to('C')

    assert converted == temp

def test_comparisons():
    assert Temperature(0, 'C') == Temperature(32, 'F')
    assert Temperature(0, 'C') < Temperature(100, 'C')
    assert Temperature(100, 'C') > Temperature(32, 'F')
    assert Temperature(0, 'C') <= Temperature(32, 'F')
    assert Temperature(100, 'C') >= Temperature(212, 'F')
    assert Temperature(0, 'C') != Temperature(100, 'C')

def test_str():
    temp = Temperature(25, 'C')
    assert str(temp) == '25.00 °C'

def test_repr():
    temp = Temperature(25, 'C')
    assert repr(temp) == 'Temperature(25, C)'

---

## Test Results

The `Temperature` class was tested using **pytest**. All 8 tests passed successfully.

| Test | Result |
|---|---|
| Initialization | <span style="color:green">✓ PASSED</span> |
| Read-only properties | <span style="color:green">✓ PASSED</span> |
| Validation | <span style="color:green">✓ PASSED</span> |
| Temperature conversions | <span style="color:green">✓ PASSED</span> |
| Round-trip conversions | <span style="color:green">✓ PASSED</span> |
| Comparisons | <span style="color:green">✓ PASSED</span> |
| String formatting (`str`) | <span style="color:green">✓ PASSED</span> |
| Representation formatting (`repr`) | <span style="color:green">✓ PASSED</span> |

**Summary:** 8 passed in 0.09 seconds.

All tests passed, confirming that the class behaves as expected for the tested functionality.

---

## Task 2: *Fractions*

For Task 2, we are writing a `Fraction` class where the numerator and denominator are integers, and the denominator is strictly positive. The work is split in two parts:

1. **2.1** - *TODO: initialiser and all six comparisons (`__eq__`, `__ne__`, `__lt__`, `__le__`, `__gt__`, `__ge__`).*
2. **2.2** - *TODO: at least four other special methods (e.g. `__add__`, `__sub__`, `__mul__`, `__truediv__`).*

*TODO: Short note on what the two parts agreed on (attribute names, whether fractions are always stored in lowest terms).*

---

### 2.1 *Initialiser and comparisons*

*TODO: Explain what this part of the class does, step by step:*

1. `__init__` - *TODO: what the method does.*
2. `__eq__ / __ne__` - *TODO: what the method does.*
3. `__lt__ / __le__ / __gt__ / __ge__` - *TODO: what the method does.*

In [ ]:
class Fraction:
    """
    TODO: One-line summary of what the class represents.
    """

    # TODO: Add the methods for 2.1 here, each with type hints and a docstring.
    ...

---

### 2.2 *Other special methods*

*TODO: Explain what this part of the class does, step by step:*

1. `__add__` - *TODO: what the method does.*
2. `__sub__` - *TODO: what the method does.*
3. `__mul__` - *TODO: what the method does.*
4. `__truediv__` - *TODO: what the method does.*

In [ ]:
class Fraction:
    """
    TODO: One-line summary of what the class represents.
    """

    # TODO: Add the methods for 2.2 here, each with type hints and a docstring.
    ...

---

### Testing with pytest

*TODO: Describe what the tests in `test_fraction.py` check (small hand-made examples where the result is easy to check by hand).*

In [ ]:
!pytest -v test_fraction.py

---

### Results and discussion of Task 2

*TODO: What the tests showed, whether the results match what we expected, and anything worth noting (edge cases, limitations, assumptions).*

---

# Use of Generative AI

### Task 1

Asked AI about how to structure the comparison methods and to help with the pytest file.

Asked for help with structuring the results markdown

---

### Task 2

**Questions about the code**
* *TODO*

**Code written by AI**
* *TODO (remove if none)*

**Testing and discussion**
* *TODO*

**Errors and corrections**
*TODO*

**What I did myself**
*TODO*

---